# Analisi statistica F1

L'utilizzo delle API di **`OpenF1`** è consigliato per l'uso di dati recenti e legati alla **telemetria**, mentre per le API di **`F1API`** sono dati **storici** risalenti fino alla prima gara nel 1950. I dati di **OpenF1** Vengono aggiornati alla mezzanotte (UTC) della fine dell'evento. Mentre per **F1API** I dati vengono aggiornati con i risultati dopo 24/48h dal termine dell'evento.

In [10]:
from urllib.request import urlopen
from urllib.parse import urlencode
import json
import pandas as pd
import API_Manager
from API_Manager import BASE_URL_OPENF1, get_data

Osserviamo la lista di tutte le gare disputate nel campionato 2025:

In [11]:
params = {
    "year": 2025,
    "session_type": "Race",
    "session_name": "Race"
}

df_sessions_2025 = get_data("sessions", params, "openf1")
df_sessions_2025[[ "circuit_short_name", "country_name", "country_code", "year" ]]

,circuit_short_name,country_name,country_code,year
0,Melbourne,Australia,AUS,2025
1,Shanghai,China,CHN,2025
2,Suzuka,Japan,JPN,2025
3,Sakhir,Bahrain,BRN,2025
4,Jeddah,Saudi Arabia,KSA,2025
5,Miami,United States,USA,2025
6,Imola,Italy,ITA,2025
7,Monte Carlo,Monaco,MON,2025
8,Catalunya,Spain,ESP,2025
9,Montreal,Canada,CAN,2025


## Posizionamento di Leclerc in Azerbaijan in cariera

Troviamo tutte le gare che si sono disutate in Azerbaijan dal 2018 ad oggi.

In [12]:
from datetime import date
print(date.today().year)

2026


In [13]:
from datetime import date
# Troviamo gli id di Azerbaijan dal 2018 in poi
params = {
    "year>": "2018",
    "year<": f"{date.today().year}",
    "session_type": "Race",
    "session_name": "Race",
    "country_code": "AZE"
}

df_AZE_sessions = get_data("sessions", params, "openf1")
df_AZE_sessions[[
    "session_key",
    "circuit_short_name",
    "country_code",
    "country_name",
    "location",
    "year"
]]

,session_key,circuit_short_name,country_code,country_name,location,year
0,9070,Baku,AZE,Azerbaijan,Baku,2023
1,9598,Baku,AZE,Azerbaijan,Baku,2024
2,9904,Baku,AZE,Azerbaijan,Baku,2025
3,11377,Baku,AZE,Azerbaijan,Baku,2026


In [14]:
params = {
    "driver_number": "16"
}

df_Leclerc_career = get_data("session_result", params, "openf1")
df_Leclerc_career[df_Leclerc_career['session_key'].isin(df_AZE_sessions['session_key'])] \
    .sort_values(by='session_key', ascending=False) \
    [['position', 'gap_to_leader', 'points']]

,position,gap_to_leader,points
426,4.0,14.136,12.0
312,9.0,36.774,2.0
198,2.0,10.91,18.0
21,3.0,21.217,15.0


## Prova ricerca tramite F1API

In [15]:
params = {
    "q": "Charles"
}
df_Leclerc_career = get_data("drivers/search", params, "f1api")
df_Leclerc_career

,driverId,name,surname,nationality,birthday,number,shortName,url
0,leclerc,Charles,Leclerc,Monaco,16/10/1997,16.0,LEC,https://en.wikipedia.org/wiki/Charles_Leclerc
1,pic,Charles,Pic,France,1990-02-15,NaN,PIC,http://en.wikipedia.org/wiki/Charles_Pic
2,pozzi,Charles,Pozzi,France,1909-08-27,NaN,None,http://en.wikipedia.org/wiki/Charles_Pozzi
3,tornaco,Charles,de Tornaco,Belgium,1927-06-07,NaN,None,http://en.wikipedia.org/wiki/Charles_de_Tornaco


In [16]:
params = {}
df_Leclerc_career = get_data("drivers/search", params, "f1api")
df_Leclerc_career

,driverId,name,surname,nationality,birthday,number,shortName,url
0,Cannoc,John,Cannon,Canada,1933-06-21,NaN,None,http://en.wikipedia.org/wiki/John_Cannon_(auto...
1,Changy,Alain,de Changy,Belgium,1922-02-05,NaN,None,http://en.wikipedia.org/wiki/Alain_de_Changy
2,abate,Carlo,Abate,Italy,1932-07-10,NaN,None,http://en.wikipedia.org/wiki/Carlo_Mario_Abate
3,abecassis,George,Abecassis,Great Britain,1913-03-21,NaN,None,http://en.wikipedia.org/wiki/George_Abecassis
4,acheson,Kenny,Acheson,Great Britain,1957-11-27,NaN,None,http://en.wikipedia.org/wiki/Kenny_Acheson
5,adamich,Andrea,de Adamich,Italy,1941-10-03,NaN,None,http://en.wikipedia.org/wiki/Andrea_de_Adamich
6,adams,Philippe,Adams,Belgium,1969-11-19,NaN,None,http://en.wikipedia.org/wiki/Philippe_Adams
7,ader,Walt,Ader,United States,1913-12-15,NaN,None,http://en.wikipedia.org/wiki/Walt_Ader
8,adolff,Kurt,Adolff,Germany,1921-11-05,NaN,None,http://en.wikipedia.org/wiki/Kurt_Adolff
9,agabashian,Fred,Agabashian,American,1913-08-21,NaN,None,http://en.wikipedia.org/wiki/Fred_Agabashian


In [17]:
params = {
    "q": "leclerc"
}
df_Leclerc_career = get_data("drivers/search", params, "f1api")
df_Leclerc_career

,driverId,name,surname,nationality,birthday,number,shortName,url
0,arthur_leclerc,Arthur,Leclerc,Monaco,14/10/2000,NaN,LEC,https://en.wikipedia.org/wiki/Arthur_Leclerc
1,leclerc,Charles,Leclerc,Monaco,16/10/1997,16.0,LEC,https://en.wikipedia.org/wiki/Charles_Leclerc


L'ID di **Charles Leclerc** è `leclerc`.

In [18]:
df_circuits = get_data("circuits", api="f1api")
df_circuits

,circuitId,circuitName,country,city,circuitLength,lapRecord,firstParticipationYear,numberOfCorners,fastestLapDriverId,fastestLapTeamId,fastestLapYear,url
0,bahrein,Bahrein International Circuit,Bahrein,Sakhir,5412.0,1:31:447,2004.0,15.0,de_la_rosa,mclaren,2005.0,https://en.wikipedia.org/wiki/Bahrain_Internat...
1,jeddah,Jeddah Corniche Circuit,Saudi Arabia,Jedda,6174.0,1:30:734,2021.0,27.0,hamilton,mercedes,2021.0,https://en.wikipedia.org/wiki/Jeddah_Corniche_...
2,albert_park,Albert Park Circuit,Australia,Melbourne,5278.0,1:19:813,1996.0,14.0,leclerc,ferrari,2024.0,https://en.wikipedia.org/wiki/Albert_Park_Circuit
3,suzuka,Suzuka International Circuit,Japan,Suzuka,5807.0,1:30:983,1987.0,18.0,hamilton,mercedes,2019.0,https://en.wikipedia.org/wiki/Suzuka_Internati...
4,miami,Miami International Autodrome,United States,Miami,5412.0,1:29:708,2022.0,19.0,max_verstappen,red_bull,2023.0,https://en.wikipedia.org/wiki/Miami_Internatio...
5,imola,Imola Autodromo Internazionale Enzo e Dino Fer...,Italy,Imola,4909.0,1:15:484,1980.0,19.0,hamilton,mercedes,2020.0,https://en.wikipedia.org/wiki/Imola_Circuit
6,monaco,Circuit de Monaco,Monaco,Monte Carlo,3337.0,1:12:909,1950.0,19.0,hamilton,mercedes,2021.0,https://en.wikipedia.org/wiki/Circuit_de_Monaco
7,gilles_villeneuve,Circuit Gilles Villeneuve,Canada,Montreal,4361.0,1:13:078,1978.0,14.0,bottas,mercedes,2019.0,https://en.wikipedia.org/wiki/Circuit_Gilles_V...
8,montmelo,Circuit de Barcelona-Catalunya,Spain,Barcelona,4657.0,1:16:330,1991.0,14.0,max_verstappen,red_bull,2023.0,https://en.wikipedia.org/wiki/Circuit_de_Barce...
9,red_bull_ring,Red Bull Ring,Austria,Spielberg,4318.0,1:05:619,1970.0,10.0,sainz,mclaren,2020.0,https://en.wikipedia.org/wiki/Red_Bull_Ring


In [19]:
df_circuits = df_circuits[df_circuits["country"] == "Azerbaijan"]
df_circuits

,circuitId,circuitName,country,city,circuitLength,lapRecord,firstParticipationYear,numberOfCorners,fastestLapDriverId,fastestLapTeamId,fastestLapYear,url
15,baku,Baku City Circuit,Azerbaijan,Baku,6003.0,1:43:009,2017.0,20.0,leclerc,ferrari,2019.0,https://en.wikipedia.org/wiki/Baku_City_Circuit


L'ID del circuito di Baku è `baku`.

Ora troviamo tutti round in tutte le stagioni dal 2018 ad oggi in cui si è disputato il Gran Premio di **Azerbaijan**.

In [20]:
df_seasons = get_data("seasons", api="f1api")
df_seasons

,championshipId,championshipName,url,year
0,f1_2026,2026 Formula 1 World Championship,https://en.wikipedia.org/wiki/2026_Formula_One...,2026
1,f1_2025,2025 Formula 1 World Championship,https://en.wikipedia.org/wiki/2025_Formula_One...,2025
2,f1_2024,2024 Formula 1 World Championship,https://en.wikipedia.org/wiki/2024_Formula_One...,2024
3,f1_2023,2023 Formula 1 World Championship,https://en.wikipedia.org/wiki/2023_Formula_One...,2023
4,f1_2022,2022 Formula 1 World Championship,https://en.wikipedia.org/wiki/2022_Formula_One...,2022
5,f1_2021,2021 Formula 1 World Championship,https://en.wikipedia.org/wiki/2021_Formula_One...,2021
6,f1_2020,2020 Formula 1 World Championship,https://en.wikipedia.org/wiki/2020_Formula_One...,2020
7,f1_2019,2019 Formula 1 World Championship,https://en.wikipedia.org/wiki/2019_Formula_One...,2019
8,f1_2018,2018 Formula 1 World Championship,https://en.wikipedia.org/wiki/2018_Formula_One...,2018
9,f1_2017,2017 Formula 1 World Championship,https://en.wikipedia.org/wiki/2017_Formula_One...,2017


In [21]:
df_Leclerc_career_seasons = df_seasons[df_seasons["year"]>=2018]
df_Leclerc_career_seasons["championshipId"]

0    f1_2026
1    f1_2025
2    f1_2024
3    f1_2023
4    f1_2022
5    f1_2021
6    f1_2020
7    f1_2019
8    f1_2018
Name: championshipId, dtype: object

In [22]:
from datetime import datetime

current_year = datetime.now().year
years = list(range(2018, current_year + 1))
lista_df = []

for year in years:
    df_temp = get_data(str(year), api="f1api")
    df_temp["season"] = year
    lista_df.append(df_temp)

df_all = pd.concat(lista_df, ignore_index=True)


/var/folders/m9/kzj1nff97gs93x60jflrf54h0000gn/T/ipykernel_7905/1631290857.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(lista_df, ignore_index=True)


In [23]:
temp = get_data("2018/2/race", api='f1api')
temp

,season,raceId,raceName,round,date,position,points,grid,time,fastLap,...,driver.surname,driver.nationality,driver.birthday,team.teamId,team.teamName,team.nationality,team.firstAppareance,team.constructorsChampionships,team.driversChampionships,team.url
0,2018,bahrain_2018,Bahrain Grand Prix,2,2018-04-08,1,25,1,1:32:01.940,None,...,Vettel,Germany,03/07/1987,ferrari,Scuderia Ferrari,Italy,1950.0,16.0,15.0,https://en.wikipedia.org/wiki/Scuderia_Ferrari
1,2018,bahrain_2018,Bahrain Grand Prix,2,2018-04-08,2,18,3,+0.699,None,...,Bottas,Finland,1989-08-28,mercedes,Mercedes Formula 1 Team,Germany,1954.0,8.0,9.0,https://en.wikipedia.org/wiki/Mercedes-Benz_in...
2,2018,bahrain_2018,Bahrain Grand Prix,2,2018-04-08,3,15,9,+6.512,None,...,Hamilton,Great Britain,07/01/1985,mercedes,Mercedes Formula 1 Team,Germany,1954.0,8.0,9.0,https://en.wikipedia.org/wiki/Mercedes-Benz_in...
3,2018,bahrain_2018,Bahrain Grand Prix,2,2018-04-08,4,12,5,+1:02.234,None,...,Gasly,France,07/02/1996,toro_rosso,Toro Rosso,Italian,NaN,NaN,NaN,http://en.wikipedia.org/wiki/Scuderia_Toro_Rosso
4,2018,bahrain_2018,Bahrain Grand Prix,2,2018-04-08,5,10,6,+1:15.046,None,...,Magnussen,Denmark,1992-10-05,haas,Haas F1 Team,United States,2016.0,NaN,NaN,https://en.wikipedia.org/wiki/Haas_F1_Team
5,2018,bahrain_2018,Bahrain Grand Prix,2,2018-04-08,6,8,7,+1:39.024,None,...,Hulkenberg,Germany,19/08/1987,renault,Renault,French,NaN,NaN,NaN,http://en.wikipedia.org/wiki/Renault_in_Formul...
6,2018,bahrain_2018,Bahrain Grand Prix,2,2018-04-08,7,6,13,+1 Lap,None,...,Alonso,Spain,1981-07-29,mclaren,McLaren Formula 1 Team,Great Britain,1966.0,10.0,13.0,https://en.wikipedia.org/wiki/McLaren
7,2018,bahrain_2018,Bahrain Grand Prix,2,2018-04-08,8,4,14,+1 Lap,None,...,Vandoorne,Belgium,1992-03-26,mclaren,McLaren Formula 1 Team,Great Britain,1966.0,10.0,13.0,https://en.wikipedia.org/wiki/McLaren
8,2018,bahrain_2018,Bahrain Grand Prix,2,2018-04-08,9,2,17,+1 Lap,None,...,Ericsson,Sweden,1990-09-02,sauber,Sauber F1 Team,Switzerland,1993.0,NaN,NaN,https://en.wikipedia.org/wiki/Sauber_Motorsport
9,2018,bahrain_2018,Bahrain Grand Prix,2,2018-04-08,10,1,8,+1 Lap,None,...,Ocon,France,17/09/1996,force_india,Force India,Indian,NaN,NaN,NaN,http://en.wikipedia.org/wiki/Racing_Point_Forc...


In [24]:
print(years[-1:])

[2026]


In [25]:
# df_all[['raceId', 'raceName', 'round', 'schedule.race.date']]
df_all.columns

Index(['raceId', 'championshipId', 'raceName', 'laps', 'round', 'url',
       'winner', 'teamWinner', 'schedule.race.date', 'schedule.race.time',
       'schedule.qualy.date', 'schedule.qualy.time', 'schedule.fp1.date',
       'schedule.fp1.time', 'schedule.fp2.date', 'schedule.fp2.time',
       'schedule.fp3.date', 'schedule.fp3.time', 'schedule.sprintQualy.date',
       'schedule.sprintQualy.time', 'schedule.sprintRace.date',
       'schedule.sprintRace.time', 'fast_lap.fast_lap',
       'fast_lap.fast_lap_driver_id', 'fast_lap.fast_lap_team_id',
       'circuit.circuitId', 'circuit.circuitName', 'circuit.country',
       'circuit.city', 'circuit.circuitLength', 'circuit.lapRecord',
       'circuit.firstParticipationYear', 'circuit.corners',
       'circuit.fastestLapDriverId', 'circuit.fastestLapTeamId',
       'circuit.fastestLapYear', 'circuit.url', 'season', 'winner.driverId',
       'winner.name', 'winner.surname', 'winner.country', 'winner.birthday',
       'winner.number', 'wi

In [26]:
df_bahrain = df_all[df_all['raceId'].str.contains("bahrain", case=False, na=False)]
df_bahrain[[ 'raceId', 'championshipId', 'raceName', 'round', 'schedule.race.date' ]]

,raceId,championshipId,raceName,round,schedule.race.date
1,bahrain_2018,f1_2018,Bahrain Grand Prix,2,2018-04-08
22,bahrain_2019,f1_2019,Bahrain Grand Prix,2,2019-03-31
56,Bahrain Grand Prix_2020,f1_2020,Bahrain Grand Prix,15,2020-11-29
59,Bahrain Grand Prix_2021,f1_2021,Bahrain Grand Prix,1,2021-03-28
81,Bahrain Grand Prix_2022,f1_2022,Bahrain Grand Prix,1,2022-03-20
103,Bahrain Grand Prix_2023,f1_2023,Bahrain Grand Prix,1,2023-03-05
152,bahrain_2025,f1_2025,Bahrain Grand Prix 2025,4,2025-04-13
188,bahrain_2026,f1_2026,Formula 1 Gulf Air Bahrain Grand Prix in Malay...,16,2026-10-04


In [27]:
# Cella 1: Filtro completo dei weekend in Bahrain
df_bahrain = df_all[
    df_all['circuit.circuitId'].str.contains("bahr", case=False, na=False) |
    df_all['raceId'].str.contains("bahr|sakhir", case=False, na=False)
].sort_values(by=['season', 'round']).copy()

df_bahrain[['season', 'round', 'raceName', 'raceId', 'schedule.race.date']]


,season,round,raceName,raceId,schedule.race.date
1,2018,2,Bahrain Grand Prix,bahrain_2018,2018-04-08
22,2019,2,Bahrain Grand Prix,bahrain_2019,2019-03-31
56,2020,15,Bahrain Grand Prix,Bahrain Grand Prix_2020,2020-11-29
57,2020,16,Sakhir Grand Prix,Sakhir Grand Prix_2020,2020-12-06
59,2021,1,Bahrain Grand Prix,Bahrain Grand Prix_2021,2021-03-28
81,2022,1,Bahrain Grand Prix,Bahrain Grand Prix_2022,2022-03-20
103,2023,1,Bahrain Grand Prix,Bahrain Grand Prix_2023,2023-03-05
125,2024,1,Gulf Air Bahrain Grand Prix 2024,bahrein_2024,2024-03-02
152,2025,4,Bahrain Grand Prix 2025,bahrain_2025,2025-04-13
188,2026,16,Formula 1 Gulf Air Bahrain Grand Prix in Malay...,bahrain_2026,2026-10-04


In [28]:
# Lista delle sessioni possibili di un weekend di gara
session_types = ["fp1", "fp2", "fp3", "qualy", "race", "sprint/qualy", "sprint/race"]

leclerc_results = []

# Iteriamo su ciascun GP del Bahrain identificato
for _, race in df_bahrain.iterrows():
    season = race['season']
    round_num = race['round']
    race_name = race['raceName']
    
    # Escludiamo eventuali gare future non ancora disputate
    if str(season) >= "2026":
        continue
        
    print(f"Scaricamento sessioni: {race_name} {season} (Round {round_num})...")
    
    for session in session_types:
        endpoint = f"{season}/{round_num}/{session}"
        try:
            df_sess = get_data(endpoint, api="f1api")
            
            # Filtriamo le righe relative a Charles Leclerc
            mask_lec = (
                (df_sess.get('driver.driverId', pd.Series(dtype=object)).str.lower() == 'leclerc') |
                (df_sess.get('driverId', pd.Series(dtype=object)).str.lower() == 'leclerc')
            )
            df_lec = df_sess[mask_lec].copy()
            
            if not df_lec.empty:
                df_lec['session'] = session
                leclerc_results.append(df_lec)
        except Exception:
            # Sessione non disponibile (es. no sprint o FP non registrata)
            pass


Scaricamento sessioni: Bahrain Grand Prix 2018 (Round 2)...
Scaricamento sessioni: Bahrain Grand Prix 2019 (Round 2)...
Scaricamento sessioni: Bahrain Grand Prix 2020 (Round 15)...
Scaricamento sessioni: Sakhir Grand Prix 2020 (Round 16)...
Scaricamento sessioni: Bahrain Grand Prix 2021 (Round 1)...
Scaricamento sessioni: Bahrain Grand Prix 2022 (Round 1)...
Scaricamento sessioni: Bahrain Grand Prix 2023 (Round 1)...
Scaricamento sessioni: Gulf Air Bahrain Grand Prix 2024 2024 (Round 1)...
Scaricamento sessioni: Bahrain Grand Prix 2025 2025 (Round 4)...


In [ ]:
if leclerc_results:
    df_leclerc_bahrain = pd.concat(leclerc_results, ignore_index=True)
    
    # Uniformiamo la colonna posizione (in qualifica è 'gridPosition', in gara è 'position')
    if 'position' in df_leclerc_bahrain.columns and 'gridPosition' in df_leclerc_bahrain.columns:
        df_leclerc_bahrain['pos_finale'] = df_leclerc_bahrain['position'].combine_first(df_leclerc_bahrain['gridPosition'])
    elif 'position' in df_leclerc_bahrain.columns:
        df_leclerc_bahrain['pos_finale'] = df_leclerc_bahrain['position']
    else:
        df_leclerc_bahrain['pos_finale'] = df_leclerc_bahrain['gridPosition']
    
    # Colonne principali di interesse
    cols_display = [
        'season', 'round', 'raceName', 'session', 
        'pos_finale', 'points', 'time', 'q1', 'q2', 'q3', 'team.teamName'
    ]
    cols_presenti = [c for c in cols_display if c in df_leclerc_bahrain.columns]
    
    df_leclerc_bahrain_view = df_leclerc_bahrain[cols_presenti]
    display(df_leclerc_bahrain_view)
else:
    print("Nessun dato trovato.")


,season,round,raceName,session,pos_finale,points,time,q1,q2,q3,team.teamName
0,2018,2,Bahrain Grand Prix,qualy,19.0,NaN,NaN,1:31.420,None,None,Sauber F1 Team
1,2018,2,Bahrain Grand Prix,race,12,0.0,+1 Lap,NaN,NaN,NaN,Sauber F1 Team
2,2019,2,Bahrain Grand Prix,qualy,1.0,NaN,NaN,1:28.495,1:28.046,1:27.866,Scuderia Ferrari
3,2019,2,Bahrain Grand Prix,race,3,16.0,+6.131,NaN,NaN,NaN,Scuderia Ferrari
4,2020,15,Bahrain Grand Prix,qualy,12.0,NaN,NaN,1:29.137,1:29.165,None,Scuderia Ferrari
5,2020,15,Bahrain Grand Prix,race,10,1.0,+1 Lap,NaN,NaN,NaN,Scuderia Ferrari
6,2020,16,Sakhir Grand Prix,qualy,4.0,NaN,NaN,0:54.249,0:53.825,0:53.613,Scuderia Ferrari
7,2020,16,Sakhir Grand Prix,race,20,0.0,Collision,NaN,NaN,NaN,Scuderia Ferrari
8,2021,1,Bahrain Grand Prix,qualy,4.0,NaN,NaN,1:30.691,1:30.010,1:29.678,Scuderia Ferrari
9,2021,1,Bahrain Grand Prix,race,6,8.0,+59.090,NaN,NaN,NaN,Scuderia Ferrari


In [33]:
view = df_leclerc_bahrain_view.groupby(by=['season', 'round'])
view